# 🥉 Bronze | Vigimed MEDICAMENTOS

Raw Data
as is production, low quality

In [14]:
%run ../config/bootstrap.py

import pandas as pd 
from pathlib import Path
from utils import get_project_root 
project_root = get_project_root() 


In [15]:
path = Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO'],
      dtype='object')

In [11]:
bronze.head(25)

,IDENTIFICACAO_NOTIFICACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,CODIGO_ATC,DETENTOR_REGISTRO,CONCENTRACAO,COMPONENTE_SUSPEITO,ACAO_ADOTADA,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,INDICACAO_MEDDRA,INDICACAO_RELATADA_NOTIFICADOR_INICIAL,DOSE,FREQUENCIA_DOSE,POSOLOGIA,DURACAO,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_MAE_PAI,NUMELO_LOTE,ATUALIZADO
0,BR-ANVISA-300000004,Suspeito,Oxacilina,Oxacillin sodium,J01CF,Teuto,None,None,None,None,None,None,250 milligram (mg),6 hora,None,4 dia,20181124,None,None,infusão intravenosa,None,5833018,2025-11-17
1,BR-ANVISA-300000005,Suspeito,Paracemol,Paracetamol,N02BE,None,None,None,Retirada do medicamento,None,None,None,None,None,None,None,20181122,20181122,None,oral,None,None,2025-11-17
2,BR-ANVISA-300000007,Suspeito,Diamox,Acetazolamide sodium,C03|N03AX|S01EC,None,None,None,Retirada do medicamento,None,None,None,250 milligram (mg),6 hora,250mg a cada 6 horas,None,20181103,20181115,None,oral,None,None,2025-11-17
3,BR-ANVISA-300000007,Suspeito,Hidantal,Phenytoin,N03AB,None,None,None,Retirada do medicamento,None,None,None,None,None,100mg a cada 8 horas,None,20181016,20181115,None,oral,None,None,2025-11-17
4,BR-ANVISA-300000008,Suspeito,Nitroglicerina,Glyceryl trinitrate,C01DA|C05AE,Cristália,None,None,None,None,None,None,None,None,None,None,20181024,None,None,infusão intravenosa,None,None,2025-11-17
5,BR-ANVISA-300000010,Suspeito,Macrodantina,Nitrofurantoin,J01XE,None,None,None,None,None,Infecção urinária,None,None,None,None,None,None,None,None,None,None,None,2025-11-17
6,BR-ANVISA-300000011,Suspeito,Dexa-Citoneurin,Cyanocobalamin|Dexamethasone|Pyridoxine hydrochloride|Thiamine hydrochloride,H02BX,Merck,None,None,Retirada do medicamento,None,None,não relatou,None,None,None,2 dia,20181126,None,None,intramuscular,None,None,2025-11-17
7,BR-ANVISA-300000012,Suspeito,Benzetacil,Benzathine benzylpenicillin,J01CE,Eurofarma,None,None,None,None,None,None,1200000 international unit ([iU]),None,None,None,20181017,None,solução injetável,intramuscular,None,503570A,2025-11-17
8,BR-ANVISA-300000014,Suspeito,Reuquinol,Hydroxychloroquine sulfate,P01BA,APSEN FARMACEUTICA S/A,None,None,Retirada do medicamento,None,Distúrbio autoimune,Doença autoimune,None,None,"400mg, 1 por dia",9 mês,20170610,None,None,oral,None,None,2025-11-17
9,BR-ANVISA-300000015,Suspeito,Anfotericina b,Amphotericin b,J02AA,TEVA,None,None,Sem alteração da dose,None,Paracoccidioidomicose,diagnóstico de paracoccidioidomicose,None,None,None,None,None,None,None,intravenosa (não especificado),None,None,2025-11-17


In [12]:
# Verificar quais colunas apresentam padrões de agrupamento (X000D ou |)
import re

def verificar_padroes_agrupamento(df):
    """
    Verifica quais colunas contêm padrões de agrupamento:
    - Pipe simples: |
    - X000D (com ou sem pipe antes)
    - _X000D_
    """
    resultados = []
    
    for col in df.columns:
        # Converter para string e verificar padrões
        valores_str = df[col].fillna('').astype(str)
        
        # Verificar cada padrão
        tem_pipe = valores_str.str.contains('\|', na=False, regex=True).any()
        tem_x000d = valores_str.str.contains('X000D', na=False, regex=False).any()
        tem_x000d_com_pipe = valores_str.str.contains('\|\s*X000D|X000D\s*\|', na=False, regex=True).any()
        tem_underscore_x000d = valores_str.str.contains('_X000D_', na=False, regex=False).any()
        
        # Contar quantas linhas têm cada padrão
        count_pipe = valores_str.str.contains('\|', na=False, regex=True).sum()
        count_x000d = valores_str.str.contains('X000D', na=False, regex=False).sum()
        
        # Verificar se tem algum padrão
        tem_qualquer_padrao = tem_pipe or tem_x000d
        
        if tem_qualquer_padrao:
            resultados.append({
                'Coluna': col,
                'Tem Pipe (|)': tem_pipe,
                'Qtd linhas com Pipe': count_pipe,
                'Tem X000D': tem_x000d,
                'Qtd linhas com X000D': count_x000d,
                'Tem |X000D ou X000D|': tem_x000d_com_pipe,
                'Tem _X000D_': tem_underscore_x000d,
            })
    
    return pd.DataFrame(resultados)

# Executar verificação
print("🔍 Verificando padrões de agrupamento nas colunas...\n")
resultados = verificar_padroes_agrupamento(bronze)

if len(resultados) > 0:
    print(f"✅ Encontradas {len(resultados)} colunas com padrões de agrupamento:\n")
    print(resultados.to_string(index=False))
    
    # Mostrar exemplos das colunas que têm mais ocorrências
    print("\n\n📊 Top 10 colunas com mais ocorrências de pipe (|):")
    top_pipe = resultados.nlargest(10, 'Qtd linhas com Pipe')[['Coluna', 'Qtd linhas com Pipe']]
    print(top_pipe.to_string(index=False))
    
    print("\n\n📊 Top 10 colunas com mais ocorrências de X000D:")
    top_x000d = resultados.nlargest(10, 'Qtd linhas com X000D')[['Coluna', 'Qtd linhas com X000D']]
    print(top_x000d.to_string(index=False))
    
    # Mostrar exemplos de valores
    print("\n\n📝 Exemplos de valores com padrões:")
    for idx, row in resultados.head(5).iterrows():
        col = row['Coluna']
        print(f"\n{col}:")
        # Pegar alguns exemplos
        valores_com_padrao = bronze[col].fillna('').astype(str)
        mask = valores_com_padrao.str.contains('\||X000D', na=False, regex=True)
        exemplos = bronze.loc[mask, col].dropna().head(3)
        for ex in exemplos:
            print(f"  - {repr(ex)}")
else:
    print("❌ Nenhuma coluna encontrada com padrões de agrupamento.")


🔍 Verificando padrões de agrupamento nas colunas...

✅ Encontradas 10 colunas com padrões de agrupamento:

                                      Coluna  Tem Pipe (|)  Qtd linhas com Pipe  Tem X000D  Qtd linhas com X000D  Tem |X000D ou X000D|  Tem _X000D_
                   PRINCIPIOS_ATIVOS_WHODRUG          True                34355      False                     0                 False        False
                                  CODIGO_ATC          True                92928      False                     0                 False        False
                           DETENTOR_REGISTRO          True                   13      False                     0                 False        False
                                CONCENTRACAO          True                   16      False                     0                 False        False
PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO          True                 3142      False                     0                 False        False
     

In [ ]:
bronze.con

## Colunas Input Grupo

In [ ]:
cols_notificacao_agg = []

## Funcao Desagrupar

In [7]:
# Desagrupar colunas com pipe "|"
# As colunas NOME_MEDICAMENTO_WHODRUG, PRINCIPIOS_ATIVOS_WHODRUG e CODIGO_ATC
# podem ter valores separados por pipe, indicando agrupamento

def desagrupar_colunas_pipe(df, colunas_agrupadas):
    """
    Desagrupa colunas que contêm valores separados por pipe "|".
    Cria uma linha para cada valor na mesma posição, mantendo as demais colunas iguais.
    
    Exemplo:
    - Se uma linha tem NOME="A|B", PRINCIPIOS="1|2", CODIGO="X|Y"
    - Serão criadas 2 linhas: (A,1,X) e (B,2,Y)
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame com dados agrupados
    colunas_agrupadas : list
        Lista com nomes das colunas que devem ser desagrupadas
        
    Returns:
    --------
    pd.DataFrame
        DataFrame desagrupado
    """
    import pandas as pd
    import re
    
    df_result = df.copy()
    colunas_agrupadas = [col for col in colunas_agrupadas if col in df_result.columns]
    
    if not colunas_agrupadas:
        return df_result
    
    # Função para processar cada linha
    def processar_linha(row):
        # Dividir cada coluna agrupada por pipe
        valores_divididos = {}
        max_len = 1
        
        for col in colunas_agrupadas:
            valor = row[col]
            if pd.isna(valor):
                valores_divididos[col] = ['']
            else:
                # Normalizar diferentes padrões de separação
                valor_str = str(valor)
                # Substituir |X000D por |
                valor_str = valor_str.replace('|X000D', '|')
                # Substituir X000D (sem pipe antes) por |
                valor_str = valor_str.replace('X000D', '|')
                # Substituir _X000D_ por |
                valor_str = valor_str.replace('_X000D_', '|')
                # Limpar pipes duplicados
                valor_str = re.sub(r'\|{2,}', '|', valor_str)
                
                # Dividir por pipe e limpar
                valores = [v.strip() for v in valor_str.split('|') if v.strip()]
                if not valores:
                    valores = ['']
                valores_divididos[col] = valores
                max_len = max(max_len, len(valores))
        
        # Garantir que todas as listas tenham o mesmo tamanho
        # Se uma lista for menor, repetir o último valor
        for col in colunas_agrupadas:
            lista = valores_divididos[col]
            if len(lista) < max_len:
                # Repetir o último valor ou usar vazio
                ultimo_valor = lista[-1] if lista else ''
                valores_divididos[col].extend([ultimo_valor] * (max_len - len(lista)))
        
        # Criar uma lista de dicionários, um para cada linha desagrupada
        linhas_desagrupadas = []
        for i in range(max_len):
            nova_linha = row.to_dict()
            for col in colunas_agrupadas:
                nova_linha[col] = valores_divididos[col][i]
            linhas_desagrupadas.append(nova_linha)
        
        return linhas_desagrupadas
    
    # Aplicar processamento a cada linha
    todas_linhas = []
    for idx, row in df_result.iterrows():
        linhas = processar_linha(row)
        todas_linhas.extend(linhas)
    
    # Criar novo DataFrame
    df_desagrupado = pd.DataFrame(todas_linhas)
    
    # Remover linhas onde todas as colunas agrupadas estão vazias
    mask_vazias = df_desagrupado[colunas_agrupadas].apply(
        lambda row: all(str(val).strip() == '' for val in row), axis=1
    )
    df_desagrupado = df_desagrupado[~mask_vazias]
    
    # Resetar índice
    df_desagrupado = df_desagrupado.reset_index(drop=True)
    
    return df_desagrupado

# Aplicar desagregação
colunas_para_desagrupar = [
    'NOME_MEDICAMENTO_WHODRUG',
    'PRINCIPIOS_ATIVOS_WHODRUG',
    'CODIGO_ATC'
]

print(f"Shape antes da desagregação: {bronze.shape}")
bronze_desagrupado = desagrupar_colunas_pipe(bronze, colunas_para_desagrupar)
print(f"Shape depois da desagregação: {bronze_desagrupado.shape}")
print(f"\nLinhas criadas: {len(bronze_desagrupado) - len(bronze)}")

# Atualizar bronze com dados desagrupados
bronze = bronze_desagrupado.copy()

# Verificar se ainda há pipes
print("\nVerificando se ainda há pipes nas colunas desagrupadas:")
for col in colunas_para_desagrupar:
    if col in bronze.columns:
        mask_com_pipe = bronze[col].astype(str).str.contains('|', na=False, regex=False)
        count = mask_com_pipe.sum()
        if count > 0:
            print(f"⚠️  {col}: {count} valores ainda contêm pipes")
            print(bronze.loc[mask_com_pipe, col].head(3))
        else:
            print(f"✅ {col}: sem pipes restantes")


Shape antes da desagregação: (552887, 23)
Shape depois da desagregação: (800744, 23)

Linhas criadas: 247857

Verificando se ainda há pipes nas colunas desagrupadas:
✅ NOME_MEDICAMENTO_WHODRUG: sem pipes restantes
✅ PRINCIPIOS_ATIVOS_WHODRUG: sem pipes restantes
✅ CODIGO_ATC: sem pipes restantes


In [13]:
bronze.query("IDENTIFICACAO_NOTIFICACAO=='BR-ANVISA-300000032'")

,IDENTIFICACAO_NOTIFICACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,CODIGO_ATC,DETENTOR_REGISTRO,CONCENTRACAO,COMPONENTE_SUSPEITO,ACAO_ADOTADA,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,INDICACAO_MEDDRA,INDICACAO_RELATADA_NOTIFICADOR_INICIAL,DOSE,FREQUENCIA_DOSE,POSOLOGIA,DURACAO,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_MAE_PAI,NUMELO_LOTE,ATUALIZADO
21,BR-ANVISA-300000032,Suspeito,Cetoprofeno,Ketoprofen,M01AE|M02AA,None,None,None,None,None,None,None,None,None,"100mg, 12/12h",None,None,None,None,intravenosa (não especificado),None,None,2025-11-17
22,BR-ANVISA-300000032,Suspeito,Gentamicin,Gentamicin,D06AX|J01GB|S01AA|S02AA|S03AA,None,None,None,None,None,None,None,240 milligram (mg),1 dia,"240mg, 24/24h",None,None,None,None,intravenosa (não especificado),None,None,2025-11-17
23,BR-ANVISA-300000032,Suspeito,Ondansetron,Ondansetron,A04AA,None,None,None,None,None,None,None,4 milligram (mg),6 hora,"4 mg, 6/6h",None,None,None,None,intravenosa (não especificado),None,None,2025-11-17
24,BR-ANVISA-300000032,Suspeito,Dipirona,Metamizole sodium,N02BB,None,None,None,None,None,None,None,None,None,"1g, 6/6h",None,None,None,None,intravenosa (não especificado),None,None,2025-11-17
25,BR-ANVISA-300000032,Suspeito,Indapamida,Indapamide,C03BA,None,None,None,None,None,None,None,1.5 milligram (mg),1 dia,"1,5mg, 24/24h",None,None,None,None,oral,None,None,2025-11-17


In [8]:
bronze.query("IDENTIFICACAO_NOTIFICACAO=='BR-ANVISA-300000032'")

,IDENTIFICACAO_NOTIFICACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,CODIGO_ATC,DETENTOR_REGISTRO,CONCENTRACAO,COMPONENTE_SUSPEITO,ACAO_ADOTADA,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,INDICACAO_MEDDRA,INDICACAO_RELATADA_NOTIFICADOR_INICIAL,DOSE,FREQUENCIA_DOSE,POSOLOGIA,DURACAO,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_MAE_PAI,NUMELO_LOTE,ATUALIZADO
28,BR-ANVISA-300000032,Suspeito,Cetoprofeno,Ketoprofen,M01AE,None,None,None,None,None,None,None,None,None,"100mg, 12/12h",None,None,None,None,intravenosa (não especificado),None,None,2025-11-17
29,BR-ANVISA-300000032,Suspeito,Cetoprofeno,Ketoprofen,M02AA,None,None,None,None,None,None,None,None,None,"100mg, 12/12h",None,None,None,None,intravenosa (não especificado),None,None,2025-11-17
30,BR-ANVISA-300000032,Suspeito,Gentamicin,Gentamicin,D06AX,None,None,None,None,None,None,None,240 milligram (mg),1 dia,"240mg, 24/24h",None,None,None,None,intravenosa (não especificado),None,None,2025-11-17
31,BR-ANVISA-300000032,Suspeito,Gentamicin,Gentamicin,J01GB,None,None,None,None,None,None,None,240 milligram (mg),1 dia,"240mg, 24/24h",None,None,None,None,intravenosa (não especificado),None,None,2025-11-17
32,BR-ANVISA-300000032,Suspeito,Gentamicin,Gentamicin,S01AA,None,None,None,None,None,None,None,240 milligram (mg),1 dia,"240mg, 24/24h",None,None,None,None,intravenosa (não especificado),None,None,2025-11-17
33,BR-ANVISA-300000032,Suspeito,Gentamicin,Gentamicin,S02AA,None,None,None,None,None,None,None,240 milligram (mg),1 dia,"240mg, 24/24h",None,None,None,None,intravenosa (não especificado),None,None,2025-11-17
34,BR-ANVISA-300000032,Suspeito,Gentamicin,Gentamicin,S03AA,None,None,None,None,None,None,None,240 milligram (mg),1 dia,"240mg, 24/24h",None,None,None,None,intravenosa (não especificado),None,None,2025-11-17
35,BR-ANVISA-300000032,Suspeito,Ondansetron,Ondansetron,A04AA,None,None,None,None,None,None,None,4 milligram (mg),6 hora,"4 mg, 6/6h",None,None,None,None,intravenosa (não especificado),None,None,2025-11-17
36,BR-ANVISA-300000032,Suspeito,Dipirona,Metamizole sodium,N02BB,None,None,None,None,None,None,None,None,None,"1g, 6/6h",None,None,None,None,intravenosa (não especificado),None,None,2025-11-17
37,BR-ANVISA-300000032,Suspeito,Indapamida,Indapamide,C03BA,None,None,None,None,None,None,None,1.5 milligram (mg),1 dia,"1,5mg, 24/24h",None,None,None,None,oral,None,None,2025-11-17


# PK

# ✅ RELACAO_MEDICAMENTO_EVENTO

In [3]:
bronze.RELACAO_MEDICAMENTO_EVENTO.value_counts(dropna=False)

RELACAO_MEDICAMENTO_EVENTO
Suspeito                        375585
Concomitante                    165566
Medicamento não administrado      8689
Interação                         2047
None                              1000
Name: count, dtype: int64

# NOME_MEDICAMENTO_WHODRUG

In [5]:
dim = bronze.NOME_MEDICAMENTO_WHODRUG.value_counts(dropna=False)
dim.head(20)

NOME_MEDICAMENTO_WHODRUG
COVID-19 vaccine AstraZeneca    14854
Comirnaty                        9635
Dipirona                         7658
CoronaVac                        6927
Cosentyx                         5978
Morfina                          5372
Vancomicina                      5362
None                             5159
Tramadol                         4832
Paclitaxel                       4792
Remicade                         4785
Infliximab                       4528
Kisqali                          3858
Omeprazol                        3729
Oxaliplatina                     3705
Ceftriaxona                      3651
Simponi                          3560
Dexametasona                     3552
Erbitux                          3375
Remsima                          3354
Name: count, dtype: int64

# PRINCIPIOS_ATIVOS_WHODRUG

In [6]:
dim = bronze.PRINCIPIOS_ATIVOS_WHODRUG.value_counts(dropna=False)
dim.head(20)

PRINCIPIOS_ATIVOS_WHODRUG
COVID-19 vaccine NRVV Ad (ChAdOx1 nCoV-19)    15048
Tozinameran                                    9716
Paclitaxel                                     8437
Infliximabe                                    8333
COVID-19 vaccine inact (Vero) CZ02             6452
Infliximab                                     6336
None                                           5649
Secukinumab                                    5386
Metamizole sodium                              5291
Tramadol                                       3893
Dipirona                                       3543
Vancomycin                                     3506
Morphine sulfate                               3263
Oxaliplatin                                    3255
Prednisone                                     3094
Oxaliplatina                                   3002
Clonazepam                                     2983
Iopromide                                      2810
Dexamethasone                         

# CODIGO_ATC

In [7]:
dim = bronze.CODIGO_ATC.value_counts(dropna=False)
dim.head(20)

CODIGO_ATC
J07BX    33962
L04AB    22915
L01XA    13803
N02BB    10921
L04AC    10890
N02AA     9077
V08AB     8997
A02BC     8841
L01CD     7407
C09CA     7296
L01EF     6528
L01BC     6465
L04AA     6362
N02AX     6242
A04AA     6219
C10AA     6005
B01AB     5914
J01DD     5594
J01CR     5455
None      5159
Name: count, dtype: int64

# ✅ DETENTOR_REGISTRO

In [8]:
dim = bronze.DETENTOR_REGISTRO.value_counts(dropna=False)
dim.head(20)

DETENTOR_REGISTRO
None                                       426201
NOVARTIS SECTOR: PHARMA                      8539
Pfizer, Inc.                                 3705
Instituto Butantan                           3569
Pfizer                                       2742
BIOGEN                                       2502
NOT SPECIFIED                                2457
Cristália                                    2128
Bayer SA                                     2028
Amgen                                        2017
Eurofarma                                    1615
Libbs                                        1552
ABL                                          1394
Cristalia                                    1275
UCB Biopharma Ltda                           1182
Astrazeneca                                  1161
Fiocruz                                      1063
Schering do Brasil Quimica Farmaceutica      1031
CRISTALIA                                    1018
Bio-Manguinhos                  

# ✅ CONCENTRACAO

In [9]:
dim = bronze.CONCENTRACAO.value_counts(dropna=False)
dim.head(40)

CONCENTRACAO
None        460259
100mg         2452
500mg         2180
1g            1891
500 mg        1287
50mg          1237
100 mg        1201
10mg          1193
40mg           950
5 mg           916
20mg           909
1G             887
500MG          826
10mg/ml        755
5mg            749
50 mg          747
10mg/mL        723
100MG          714
300mg          686
50mg/ml        661
10 mg          628
25mg           575
150 mg         532
20 mg          509
8mg/4ml        508
400mg          507
10 mg/mL       475
40 mg          465
200mg          451
500 MG         442
1 g            421
20mg/mL        410
1000mg         409
15 mg          405
150 mg         404
10MG/ML        396
25 mg          381
4mg            380
15 mg          364
350            362
Name: count, dtype: int64

# ✅ COMPONENTE_SUSPEITO

In [10]:
dim = bronze.COMPONENTE_SUSPEITO.value_counts(dropna=False)
dim.head(20)

COMPONENTE_SUSPEITO
None                            496737
Princípio Ativo                  55897
Excipiente, não classificado       127
Corante                             37
Conservante                         32
Solvente                            24
Excesso percentual                  11
Antioxidante                         8
Agente Flavorizador                  7
Estabilizante                        7
Name: count, dtype: int64

# ✅ ACAO_ADOTADA

In [11]:
dim = bronze.ACAO_ADOTADA.value_counts(dropna=False)
dim.head(20)

ACAO_ADOTADA
None                       267347
Retirada do medicamento     85273
Desconhecido                79419
Sem alteração da dose       55618
Não aplicável               52568
Redução da dose              8202
Aumento da dose              4460
Name: count, dtype: int64

# ✅ PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO

In [12]:
dim = bronze.PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO.value_counts(dropna=False)
dim.head(20)

PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO
None                                                            534294
Erro de Medicação                                                 7969
Uso "off-label" Uso sem registro                                  3407
Lotes testados e dentro das especificações                        1286
Uso incorreto                                                      915
Superdose                                                          915
Erro de Medicação|Uso incorreto                                    474
Abuso                                                              468
Erro de Medicação|Uso "off-label" Uso sem registro                 441
Erro de Medicação_x000D_|Uso "off-label" Uso sem registro          392
Erro de Medicação_x000D_|Uso incorreto                             360
Exposição ocupacional                                              302
Uso "off-label" Uso sem registro_x000D_|Erro de Medicação          145
|                               

In [13]:
col = "PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO"

s = bronze[col].fillna("").astype(str).str.upper()

s = s.str.replace(r"X000D_\|", "|", regex=True)
s = s.str.replace(r"X000D_", "|", regex=True)
s = s.str.replace(r"_X000D_", "|", regex=True)
s = s.str.replace(r"X000D", "|", regex=True)

s = s.str.replace(r"_+", "|", regex=True)
s = s.str.replace(r"\|{2,}", "|", regex=True)
s = s.str.replace('"', "")
s = s.str.strip("| ")

# transforma em LISTA de problemas (separados por "|")
bronze[col] = s.apply(
    lambda txt: [p.strip() for p in txt.split("|") if p.strip()]
)


In [14]:
print(bronze[col].iloc[0], type(bronze[col].iloc[0]))


[] <class 'list'>


In [15]:
PROBLEMAS_unicas = (
    bronze[col]
    .explode()          # cada problema vira uma linha
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s.ne("")]   # remove strings vazias
    .drop_duplicates()
    .sort_values()
)

for g in PROBLEMAS_unicas:
    print(repr(g))


'ABUSO'
'ERRO DE MEDICAÇÃO'
'EXPOSIÇÃO OCUPACIONAL'
'FALSIFICAÇÃO'
'LOTES TESTADOS E DENTRO DAS ESPECIFICAÇÕES'
'LOTES TESTADOS E FORA DAS ESPECIFICAÇÕES'
'MEDICAMENTO TOMADO FORA DA DATA DE VALIDADE'
'MEDICAMENTO TOMADO PELO PAI'
'SUPERDOSE'
'USO INCORRETO'
'USO OFF-LABEL USO SEM REGISTRO'


# INDICACAO_MEDDRA

In [16]:
dim = bronze.INDICACAO_MEDDRA.value_counts(dropna=False)
dim.head(20)

INDICACAO_MEDDRA
None                                              325534
Uso de medicamento para indicação desconhecida     19595
Produto usado para indicação desconhecida          16515
Doença de Crohn                                     6875
Artrite reumatoide                                  6207
Hipertensão                                         4399
Profilaxia de COVID-19                              4179
Esclerose múltipla                                  4085
Mieloma múltiplo                                    3449
Espondilite anquilosante                            3139
Câncer de mama                                      3119
Dor                                                 2982
Analgesia                                           2769
Neoplasia maligna da mama                           2694
Diabetes                                            2502
Infecção                                            2269
Artrite psoriática                                  2146
Profilaxia    

# INDICACAO_RELATADA_NOTIFICADOR_INICIAL

In [17]:
dim = bronze.INDICACAO_RELATADA_NOTIFICADOR_INICIAL.value_counts(dropna=False)
dim.head(40)

INDICACAO_RELATADA_NOTIFICADOR_INICIAL
None                                         315982
Product used for unknown indication           11080
Drug use for unknown indication                2710
DRUG USE FOR UNKNOWN INDICATION                2566
Hipertensão                                    2132
Doença de Crohn                                2097
Diabetes                                       1878
Produto usado para indicação desconhecida      1769
Artrite reumatoide                             1508
COVID-1' immunisation                          1352
Profilaxia para covid-1'                       1303
Espondilite anquilosante                       1112
profilaxia covid-1'                             917
MULTIPLE SCLEROSIS                              882
Analgesia                                       853
Osteoporose                                     850
Dor                                             850
Psoriatic arthritis                             840
Psoríase                 

# ✅ DOSE

In [18]:
dim = bronze.DOSE.value_counts(dropna=False)
dim.head(40)

DOSE
None                    339554
100 milligram (mg)       10204
50 milligram (mg)         9612
10 milligram (mg)         8231
300 milligram (mg)        7323
1 dosage form ({DF})      6812
20 milligram (mg)         6795
40 milligram (mg)         6051
500 milligram (mg)        5743
1 gram (g)                5266
5 milligram (mg)          4643
150 milligram (mg)        4503
400 milligram (mg)        4425
200 milligram (mg)        4335
25 milligram (mg)         4016
1000 milligram (mg)       3744
600 milligram (mg)        2880
8 milligram (mg)          2709
2 gram (g)                2669
2 dosage form ({DF})      2601
2 milligram (mg)          2584
4 milligram (mg)          1947
30 milligram (mg)         1917
15 milligram (mg)         1762
60 milligram (mg)         1652
1 milligram (mg)          1550
3 dosage form ({DF})      1422
75 milligram (mg)         1388
80 milligram (mg)         1344
10 millilitre (mL)        1290
2.5 milligram (mg)        1220
20 microgram (ug)         1216
120

# ✅ FREQUENCIA_DOSE

In [19]:
dim = bronze.FREQUENCIA_DOSE.value_counts(dropna=False)
dim.head(40)

FREQUENCIA_DOSE
                   379079
1 dia               34451
1 por 1 dia         16549
1                    8121
12 hora              6179
 cíclico             5583
 Total               5109
2 por 12 hora        4604
1 mês                3992
8 semana             3903
2 por 1 dia          3901
None                 3583
4 semana             3553
3 por 8 hora         3550
1 semana             3285
15 dia               3161
3 por 1 dia          2824
1 Total              2755
1 por 24 hora        2658
8 hora               2611
4 por 6 hora         2419
1 cíclico            2251
2 semana             2023
4 por 1 dia          1898
24 hora              1750
2 dia                1597
6 hora               1437
1 por 1 semana       1354
 se necessário       1276
21 dia               1244
1 por 21 dia         1227
1 por 1 mês          1224
1 por 12 hora        1159
3 semana             1080
1 por 8 hora          972
2 por 2 dia           900
14 dia                868
0.5 dia               

# POSOLOGIA

In [20]:
dim = bronze.POSOLOGIA.value_counts(dropna=False)
dim.head(40)

POSOLOGIA
None                                   275637
UNK                                     19441
Desconhecido                             3083
Posologia desconhecida                   1509
1 comprimido ao dia                      1323
Não informado                            1210
NI                                       1145
primeira dose                            1005
1 comprimido por dia                      974
Frequência desconhecida                   955
Desconhecida                              913
Primeira dose                             867
1 DF, QD                                  858
1                                         788
segunda dose                              779
Dose única                                715
dose única                                674
DOSE 1, SINGLE                            632
50 mg                                     597
1 dose                                    563
100 mg                                    545
10 mg                   

# ✅ DURACAO

In [21]:
dim = bronze.DURACAO.value_counts(dropna=False)
dim.head(40)

DURACAO
None         473049
1 dia         20933
2 dia          4107
3 dia          3079
1 hora         2959
5 dia          2485
1 minuto       2274
4 dia          2253
30 minuto      2101
7 dia          1878
2 hora         1577
6 dia          1501
0 dia          1404
8 dia          1261
10 dia         1105
5 minuto       1070
10 minuto      1058
9 dia           843
20 minuto       757
15 dia          719
15 minuto       718
11 dia          709
3 hora          693
14 dia          651
12 dia          577
1 ano           576
60 minuto       576
22 dia          563
1 mês           554
3 mês           550
2 mês           544
13 dia          512
4 hora          472
90 minuto       439
21 dia          378
2 ano           374
16 dia          373
40 minuto       339
24 hora         334
6 mês           334
Name: count, dtype: int64

# 🧐 VIA_ADMINISTRACAO_MAE_PAI

In [22]:
dim = bronze.VIA_ADMINISTRACAO_MAE_PAI.value_counts(dropna=False)
dim.head(40)

VIA_ADMINISTRACAO_MAE_PAI
None                                552748
E2B R2 Code: 050 - Other                38
E2B R2 Code: 065 - Unknown              14
Oral                                    14
intravenosa (não especificado)          11
oral                                    10
Unknown                                  8
Subcutaneous                             7
Oral use                                 6
Via intravenosa                          5
desconhecida                             5
infusão intravenosa                      3
Intramuscular use                        2
subcutânea                               2
Respiratory (inhalation)                 2
intracerebral                            2
parenteral                               2
Subcutaneous use                         2
intramuscular                            1
200 miligrama, Semanalmente (qw)         1
intrauterina                             1
outra                                    1
bolus intravenoso           

# 🧐 VIA_ADMINISTRACAO

In [23]:
dim = bronze.VIA_ADMINISTRACAO.value_counts(dropna=False)
dim.head(40)

VIA_ADMINISTRACAO
None                                     307203
oral                                      31059
infusão intravenosa                       26376
intramuscular                             19866
desconhecida                              18764
intravenosa (não especificado)            18495
EV                                        11951
Unknown                                   11519
Oral                                      11327
E2B R2 Code: 065 - Unknown                 9741
subcutânea                                 8826
Intravenosa                                4249
Endovenosa                                 3855
IV                                         3797
endovenosa                                 3543
intravenosa                                3008
endovenoso                                 2749
Oral use                                   2735
E2B R2 Code: 050 - Other                   2589
ENDOVENOSA                                 2571
Intravenous (not other

# 🧐 FORMA_FARMACEUTICA

In [24]:
dim = bronze.FORMA_FARMACEUTICA.value_counts(dropna=False)
dim.head(40)

FORMA_FARMACEUTICA
None                                     400411
Tablet                                     9792
comprimido                                 7291
Solução injetável                          7177
solução injetável                          7077
Solution for injection                     6717
Comprimido                                 4215
ampola                                     3911
Film-coated tablet                         3827
AMPOLA                                     3617
Injetável                                  3084
Concentrate for solution for infusion      2903
Solution for infusion                      2863
SOLUTION FOR INJECTION                     2680
Unknown                                    2611
COMPRIMIDO                                 2425
Ampola                                     2334
injetável                                  2204
SOLUÇÃO INJETÁVEL                          1981
solução                                    1822
injetavel            